# N152 · SQL入门、过滤与NULL

**目标：** 建立表的行列语义并写出正确过滤条件。

**先修：** N007, N010。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** SELECT/WHERE/DISTINCT；ORDER BY/LIMIT；NULL三值逻辑；SQL多重集合。

## 从问题到算法

SQL表达的是集合/多重集筛选，不是逐行修改变量。WHERE只保留谓词为TRUE的行；与NULL比较得到UNKNOWN，不能用referee_id<>2保留未知推荐人，必须另写IS NULL。边界面积/人口条件采用OR；不带ORDER BY的结果不承诺顺序。

## 最小实现

**本章接口：** `create_sqlite_fixture()`、`query_recyclable_products.sql`、`query_big_countries.sql`、`query_customer_referee.sql`

In [1]:
import sqlite3

SQL_QUERIES = {
    'query_recyclable_products.sql': r'''SELECT product_id
FROM Products
WHERE low_fats = 'Y' AND recyclable = 'Y';''',
    'query_big_countries.sql': r'''SELECT name, population, area
FROM World
WHERE area >= 3000000 OR population >= 25000000;''',
    'query_customer_referee.sql': r'''SELECT name
FROM Customer
WHERE referee_id IS NULL OR referee_id <> 2;''',
}

def create_sqlite_fixture():
    connection=sqlite3.connect(':memory:')
    connection.executescript(r'''
CREATE TABLE Products(product_id INTEGER PRIMARY KEY,low_fats TEXT,recyclable TEXT);
INSERT INTO Products VALUES (0,'Y','N'),(1,'Y','Y'),(2,'N','Y'),(3,'Y','Y');
CREATE TABLE World(name TEXT PRIMARY KEY,area INTEGER,population INTEGER,gdp INTEGER);
INSERT INTO World VALUES ('AreaBoundary',3000000,1,0),('PopulationBoundary',1,25000000,0),('Small',2999999,24999999,0);
CREATE TABLE Customer(id INTEGER PRIMARY KEY,name TEXT,referee_id INTEGER);
INSERT INTO Customer VALUES (1,'Alice',NULL),(2,'Bob',2),(3,'Cara',1),(4,'Dan',NULL);
''')
    return connection

def query_rows(connection,name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 正确性、前提与复杂度

AND要求两个条件同时为TRUE，OR要求至少一个为TRUE；NULL不是某个普通值，因此需使用IS NULL谓词。查询明确只选题目要求的列，不用SELECT *导致列契约漂移。测试按集合验证无排序要求的结果，同时保留NULL语义检查。

**代价：** 无额外索引时这些过滤查询扫描O(行数)，实际执行计划由数据库决定。本章在真实SQLite内存库执行；SQL文本以.sql名称映射保存在SQL_QUERIES中。其他数据库方言未在本轮执行。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
for name in SQL_QUERIES:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
db.close()

query_recyclable_products.sql


product_id
1
3


query_big_countries.sql


name,population,area
AreaBoundary,1,3000000
PopulationBoundary,25000000,1


query_customer_referee.sql


name
Alice
Cara
Dan


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
db=create_sqlite_fixture()
assert set(query_rows(db,'query_recyclable_products.sql'))=={(1,),(3,)}
assert {row[0] for row in query_rows(db,'query_big_countries.sql')}=={'AreaBoundary','PopulationBoundary'}
assert set(query_rows(db,'query_customer_referee.sql'))=={('Alice',),('Cara',),('Dan',)}
assert db.execute('SELECT NULL <> 2, NULL = NULL, NULL IS NULL').fetchone()==(None,None,1)
db.executescript('DELETE FROM Products; DELETE FROM World; DELETE FROM Customer;')
assert all(query_rows(db,name)==[] for name in SQL_QUERIES)
db.close()
print('N152: 所有本章断言通过')

N152: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释referee_id<>2会漏掉NULL。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较去重与保留重复行。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1757. Recyclable and Low Fat Products（canonical）
- 595. Big Countries（canonical）
- 584. Find Customer Referee（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。